In [6]:
from collections import deque
import copy
import time


easy = """004030050
609400000
005100489
000060930
300807002
026040000
453009600
000004705
090050200"""

medium = """530070000
600195000
098000060
800060003
400803001
700020006
060000280
000419005
000080079"""

hard = """102040007
000080000
009500304
000607900
540000026
006405000
708003400
000010000
200060509"""

veryhard = """001007000
600400300
000030064
380076000
000000036
270015000
000020051
700100200
008009000"""

files_data = {
    "easy.txt": easy,
    "medium.txt": medium,
    "hard.txt": hard,
    "veryhard.txt": veryhard
}

for name, data in files_data.items():
    with open(name, "w") as f:
        f.write(data)


bt_calls = 0
bt_failures = 0



def read_board(filename):
    board = []

    with open(filename, "r") as f:
        for line in f:
            row = [int(x) for x in line.strip()]
            board.append(row)

    return board



def print_board(board):

    for i in range(9):

        if i == 3 or i == 6:
            print("-" * 21)

        for j in range(9):

            if j == 3 or j == 6:
                print("|", end=" ")

            if board[i][j] == 0:
                print(".", end=" ")
            else:
                print(board[i][j], end=" ")

        print()


def get_peers(r, c):

    peers = set()

    for i in range(9):

        if i != c:
            peers.add((r, i))

        if i != r:
            peers.add((i, c))

    br = (r // 3) * 3
    bc = (c // 3) * 3

    for i in range(br, br + 3):
        for j in range(bc, bc + 3):

            if (i, j) != (r, c):
                peers.add((i, j))

    return peers


def build_domains(board):

    domains = {}

    for r in range(9):
        for c in range(9):

            if board[r][c] != 0:
                domains[(r, c)] = {board[r][c]}
            else:
                domains[(r, c)] = set(range(1, 10))

    return domains



def revise(domains, xi, xj):

    revised = False

    if len(domains[xj]) == 1:

        val = next(iter(domains[xj]))

        if val in domains[xi] and len(domains[xi]) > 1:
            domains[xi].remove(val)
            revised = True

    return revised


def ac3(domains):

    queue = deque()

    for cell in domains:
        for peer in get_peers(*cell):
            queue.append((cell, peer))

    while queue:

        xi, xj = queue.popleft()

        if revise(domains, xi, xj):

            if len(domains[xi]) == 0:
                return False

            for peer in get_peers(*xi):
                if peer != xj:
                    queue.append((peer, xi))

    return True


def select_unassigned(assignment, domains):

    unassigned = []

    for cell in domains:
        if cell not in assignment:
            unassigned.append(cell)

    return min(unassigned, key=lambda x: len(domains[x]))


def forward_check(domains, var, val):

    removed = []

    for peer in get_peers(*var):

        if val in domains[peer] and len(domains[peer]) > 1:

            domains[peer].remove(val)
            removed.append((peer, val))

            if len(domains[peer]) == 0:
                return False, removed

    return True, removed

def backtrack(assignment, domains):

    global bt_calls, bt_failures

    bt_calls += 1

    if len(assignment) == 81:
        return assignment

    var = select_unassigned(assignment, domains)

    for val in sorted(domains[var]):

        valid = True

        for peer in get_peers(*var):

            if peer in assignment and assignment[peer] == val:
                valid = False
                break

        if valid:

            assignment[var] = val

            old_domains = copy.deepcopy(domains)
            domains[var] = {val}

            ok1 = ac3(domains)
            ok2, removed = forward_check(domains, var, val)

            if ok1 and ok2:

                result = backtrack(assignment, domains)

                if result:
                    return result

            del assignment[var]
            domains.clear()
            domains.update(old_domains)

    bt_failures += 1
    return None

def solve(filename):

    global bt_calls, bt_failures

    bt_calls = 0
    bt_failures = 0

    board = read_board(filename)

    print("\nPuzzle:", filename)
    print("\nInitial Board:")
    print_board(board)

    domains = build_domains(board)

    ac3(domains)

    assignment = {}

    for r in range(9):
        for c in range(9):

            if board[r][c] != 0:
                assignment[(r, c)] = board[r][c]

    start = time.time()

    result = backtrack(assignment, domains)

    end = time.time()

    if result:

        solved = [[0 for j in range(9)] for i in range(9)]

        for (r, c), val in result.items():
            solved[r][c] = val

        print("\nSolved Board:")
        print_board(solved)

    else:
        print("\nNo Solution")

    print("\nTime:", round(end - start, 4), "sec")
    print("Backtrack Calls:", bt_calls)
    print("Failures:", bt_failures)


files = ["easy.txt", "medium.txt", "hard.txt", "veryhard.txt"]

for f in files:
    solve(f)


Puzzle: easy.txt

Initial Board:
. . 4 | . 3 . | . 5 . 
6 . 9 | 4 . . | . . . 
. . 5 | 1 . . | 4 8 9 
---------------------
. . . | . 6 . | 9 3 . 
3 . . | 8 . 7 | . . 2 
. 2 6 | . 4 . | . . . 
---------------------
4 5 3 | . . 9 | 6 . . 
. . . | . . 4 | 7 . 5 
. 9 . | . 5 . | 2 . . 

Solved Board:
7 8 4 | 9 3 2 | 1 5 6 
6 1 9 | 4 8 5 | 3 2 7 
2 3 5 | 1 7 6 | 4 8 9 
---------------------
5 7 8 | 2 6 1 | 9 3 4 
3 4 1 | 8 9 7 | 5 6 2 
9 2 6 | 5 4 3 | 8 7 1 
---------------------
4 5 3 | 7 2 9 | 6 1 8 
8 6 2 | 3 1 4 | 7 9 5 
1 9 7 | 6 5 8 | 2 4 3 

Time: 0.3482 sec
Backtrack Calls: 50
Failures: 0

Puzzle: medium.txt

Initial Board:
5 3 . | . 7 . | . . . 
6 . . | 1 9 5 | . . . 
. 9 8 | . . . | . 6 . 
---------------------
8 . . | . 6 . | . . 3 
4 . . | 8 . 3 | . . 1 
7 . . | . 2 . | . . 6 
---------------------
. 6 . | . . . | 2 8 . 
. . . | 4 1 9 | . . 5 
. . . | . 8 . | . 7 9 

Solved Board:
5 3 4 | 6 7 8 | 9 1 2 
6 7 2 | 1 9 5 | 3 4 8 
1 9 8 | 3 4 2 | 5 6 7 
---------------------
8 5 9 